# Centroid + Feature Mean Tracking

Barebones adjacent-frame tracking with dense cost matrices and Hungarian assignment. Methods: centroid-only, feature-mean-only, and centroid plus a small clipped feature-mean correction. No shape terms, no search radius, no raw-intensity dependency.

In [14]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd
import torch

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "scripts" / "evaluation" / "feature_mean_tracking.py").exists():
    repo_root = repo_root.parent

module_path = repo_root / "scripts" / "evaluation" / "feature_mean_tracking.py"
if str(module_path.parent) not in sys.path:
    sys.path.insert(0, str(module_path.parent))
spec = importlib.util.spec_from_file_location("feature_mean_tracking", module_path)
feature_mean_tracking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = feature_mean_tracking
spec.loader.exec_module(feature_mean_tracking)

FeatureMeanConfig = feature_mean_tracking.FeatureMeanConfig
run_feature_mean_tracking = feature_mean_tracking.run_feature_mean_tracking
save_result = feature_mean_tracking.save_result

print(f"repo_root: {repo_root}")
print(f"cuda: {torch.cuda.is_available()} devices={torch.cuda.device_count()}")

repo_root: /nfs/scratch2/inacio/code/llsm/spatialdino/spatialdino
cuda: True devices=1


In [30]:
# Edit these paths before running.
INPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/rope/")
SEGMENTATION_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/gt/")
OUTPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/")

MAX_FRAMES = None
COMPUTE_GT_METRICS = True

METHODS = ("centroid", "feature_mean", "centroid_feature")
TRACKS_METHOD = "centroid_feature"
CENTROID_FEATURE_WEIGHT = 0.01
FEATURE_NORM_CLIP = 3.0
Z_WEIGHT = 2.5

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [31]:
config = FeatureMeanConfig(
    n_features=384,
    samples_per_object=128,
    seed=12345,
    device=DEVICE,
    methods=METHODS,
    tracks_method=TRACKS_METHOD,
    centroid_feature_weight=CENTROID_FEATURE_WEIGHT,
    feature_norm_clip=FEATURE_NORM_CLIP,
    z_weight=Z_WEIGHT,
    max_frames=MAX_FRAMES,
    compute_gt_metrics=COMPUTE_GT_METRICS,
    progress=True,
)

config

FeatureMeanConfig(n_features=384, samples_per_object=128, seed=12345, device='cuda:0', methods=('centroid', 'feature_mean', 'centroid_feature'), tracks_method='centroid_feature', centroid_feature_weight=0.01, feature_norm_clip=3.0, z_weight=2.5, max_frames=None, compute_gt_metrics=True, progress=True, object_batch_size=512, feature_channel_block=64, sample_batch_size=131072)

In [32]:
result = run_feature_mean_tracking(
    INPUT_PATH,
    SEGMENTATION_PATH,
    config=config,
)

saved_paths = save_result(result, OUTPUT_PATH)
print(json.dumps(saved_paths, indent=2))

[feature-mean-tracking] found 13 frame(s); using 384/390 feature channel(s), 128 sample(s)/object, methods=('centroid', 'feature_mean', 'centroid_feature'), device=cuda:0


feature means:   0%|                                                                                          …

[feature-mean-tracking] feature means completed in 14.32s


adjacent pairs:   0%|                                                                                         …

[feature-mean-tracking] matching and tracks completed in 0.24s; total 14.57s
{
  "tracks_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/tracks.csv",
  "assignments_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/assignments.csv",
  "timings_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/timings.csv",
  "config_json": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/config.json",
  "tracks_centroid_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_mean_tracking/tracks_centroid.csv",
  "tracks_centroid_nested_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/feature_

In [29]:
display(result.metrics if result.metrics is not None else pd.DataFrame())
display(result.pair_metrics.head(20) if result.pair_metrics is not None else pd.DataFrame())

,method,frame_pairs,trackable_count,link_pred_count,link_tp,link_fp,link_fn,precision,recall,f1,median_centroid_scale,median_feature_scale,centroid_feature_weight,feature_norm_clip
0,centroid,12,3600,3600,3162,438,438,0.878333,0.878333,0.878333,266.472183,NaN,NaN,NaN
1,feature_mean,12,3600,3600,2339,1261,1261,0.649722,0.649722,0.649722,NaN,0.077283,NaN,NaN
2,centroid_feature,12,3600,3600,3222,378,378,0.895000,0.895000,0.895000,266.472183,0.077283,0.5,3.0


,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_count,cand_count,trackable_count,link_pred_count,link_tp,link_fp,link_fn,precision,recall,f1,centroid_scale,feature_scale,centroid_feature_weight,feature_norm_clip
0,centroid,0,1,00,04,300,300,300,300,247,53,53,0.823333,0.823333,0.823333,256.756958,NaN,NaN,NaN
1,feature_mean,0,1,00,04,300,300,300,300,175,125,125,0.583333,0.583333,0.583333,NaN,0.080088,NaN,NaN
2,centroid_feature,0,1,00,04,300,300,300,300,251,49,49,0.836667,0.836667,0.836667,256.756958,0.080088,0.5,3.0
3,centroid,1,2,04,08,300,300,300,300,264,36,36,0.880000,0.880000,0.880000,259.025940,NaN,NaN,NaN
4,feature_mean,1,2,04,08,300,300,300,300,180,120,120,0.600000,0.600000,0.600000,NaN,0.078708,NaN,NaN
5,centroid_feature,1,2,04,08,300,300,300,300,259,41,41,0.863333,0.863333,0.863333,259.025940,0.078708,0.5,3.0
6,centroid,2,3,08,12,300,300,300,300,267,33,33,0.890000,0.890000,0.890000,261.065765,NaN,NaN,NaN
7,feature_mean,2,3,08,12,300,300,300,300,181,119,119,0.603333,0.603333,0.603333,NaN,0.076411,NaN,NaN
8,centroid_feature,2,3,08,12,300,300,300,300,267,33,33,0.890000,0.890000,0.890000,261.065765,0.076411,0.5,3.0
9,centroid,3,4,12,16,300,300,300,300,260,40,40,0.866667,0.866667,0.866667,262.783264,NaN,NaN,NaN


In [19]:
display(result.assignments.head(50))

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_label,assigned_cand_label,cost,is_true_link
0,centroid,0,1,00,04,1,1,0.030817,True
1,centroid,0,1,00,04,2,2,0.032573,True
2,centroid,0,1,00,04,3,3,0.104902,True
3,centroid,0,1,00,04,4,4,0.058005,True
4,centroid,0,1,00,04,5,252,0.110008,False
5,centroid,0,1,00,04,6,6,0.063326,True
6,centroid,0,1,00,04,7,7,0.028475,True
7,centroid,0,1,00,04,8,8,0.115334,True
8,centroid,0,1,00,04,9,9,0.021009,True
9,centroid,0,1,00,04,10,102,0.027259,False


In [20]:
display(result.tracks.head(50))

,track_id,start,t,x,y,z,A,track_length
0,1,1,1,309.000000,223.000000,12.000000,1.0,13
1,1,1,2,316.666656,224.079361,12.460318,1.0,13
2,1,1,3,309.758057,214.370972,9.225806,1.0,13
3,1,1,4,286.432831,225.343277,16.179104,1.0,13
4,1,1,5,289.343750,247.937500,28.500000,1.0,13
5,1,1,6,287.671875,252.109375,36.562500,1.0,13
6,1,1,7,261.671875,236.453125,47.359375,1.0,13
7,1,1,8,246.079361,244.174606,51.269840,1.0,13
8,1,1,9,228.344315,289.781433,63.452095,1.0,13
9,1,1,10,228.590912,311.818176,60.878788,1.0,13
